# 04 — TabNet kurulum ve ilk deneme

**Sahibi:** Öğrenci 3 (TabNet & Comparative Analytics)

Bu notebook'un bu bölümü plan madde 9 / Adım 1-2'yi karşılar:
- `torch` ve `pytorch_tabnet` kurulumunun çalıştığını doğrula.
- 2.000 satırlık bir örneklemle 5 epoch'luk hızlı bir deneme yap.
- Kategorik kolonları `CAT_LEVELS`'tan türetilen sabit kodlarla 0..k-1 aralığına eşle
  (veriden değil, sözleşmeden türetilir ki test setinde görülmemiş bir kod çökme yaratmasın).


In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Adım 1: Kurulum kontrolü

In [2]:
import torch
import pytorch_tabnet
from pytorch_tabnet.tab_model import TabNetClassifier
from importlib.metadata import version

print("torch         ", torch.__version__)
print("pytorch_tabnet", version("pytorch-tabnet"))
print("cuda var mı   ", torch.cuda.is_available(), "(CPU sürümüyle False beklenir)")

torch          2.14.1+cpu
pytorch_tabnet 4.1.0
cuda var mı    False (CPU sürümüyle False beklenir)


## Adım 2: Kategorik hazırlık

`cat_idxs` ve `cat_dims`, veriden değil `CAT_LEVELS` sözleşmesinden hesaplanır.
Sayısal kolonlar `make_preprocessor("tabnet")` ile ölçeklenir, kategorikler bu
hücrede 0..k-1 aralığına eşlenir (`make_preprocessor("tabnet")` kodları ham
bırakır, TabNet embedding'i 0'dan başlayan ardışık kod bekler).

In [3]:
from src.config import CAT_COLS, CAT_LEVELS, FEATURES, NUM_COLS
from src.data import load_split, to_xy
from src.features import make_preprocessor

CODE_MAPS = {col: {code: i for i, code in enumerate(levels)} for col, levels in CAT_LEVELS.items()}


def encode_categoricals(X):
    """Map each categorical column's raw codes to 0..k-1 using CAT_LEVELS."""
    X = X.copy()
    for col in CAT_COLS:
        X[col] = X[col].map(CODE_MAPS[col]).astype("int64")
    return X


cat_idxs = [FEATURES.index(col) for col in CAT_COLS]
cat_dims = [len(CAT_LEVELS[col]) for col in CAT_COLS]
list(zip(CAT_COLS, cat_idxs, cat_dims))

[('SEX', 1, 2),
 ('EDUCATION', 2, 7),
 ('MARRIAGE', 3, 4),
 ('PAY_0', 5, 11),
 ('PAY_2', 6, 11),
 ('PAY_3', 7, 11),
 ('PAY_4', 8, 11),
 ('PAY_5', 9, 11),
 ('PAY_6', 10, 11)]

## Adım 3: 2.000 satırlık hızlı deneme (5 epoch)

In [4]:
from src.config import SEED

VARIANT = "v0"  # Öğrenci 1'in clean_* dosyaları gelince "clean" olacak

train_full = load_split("train", VARIANT)
sample = train_full.sample(n=2000, random_state=SEED)
X_sample, y_sample = to_xy(sample)

preproc = make_preprocessor("tabnet")
X_scaled = preproc.fit_transform(X_sample)
X_scaled = encode_categoricals(X_scaled)
X_scaled = X_scaled[FEATURES].to_numpy(dtype="float32")
y_arr = y_sample.to_numpy()

X_scaled.shape, y_arr.shape

((2000, 23), (2000,))

In [5]:
clf = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,
    n_d=8,
    n_a=8,
    n_steps=3,
    gamma=1.3,
    seed=SEED,
    verbose=1,
)
clf.fit(
    X_scaled,
    y_arr,
    max_epochs=5,
    batch_size=256,
    virtual_batch_size=64,
)
print("Hızlı deneme tamamlandı, TabNet kurulumu çalışıyor.")

epoch 0  | loss: 0.64045 |  0:00:00s
epoch 1  | loss: 0.55545 |  0:00:00s


epoch 2  | loss: 0.52433 |  0:00:00s
epoch 3  | loss: 0.51859 |  0:00:00s


epoch 4  | loss: 0.49852 |  0:00:00s
Hızlı deneme tamamlandı, TabNet kurulumu çalışıyor.


## Adım 3: Grid tuning (Gün 2-4)

Taranan parametreler:
- `n_d = n_a` ∈ {8, 16, 32}
- `n_steps` ∈ {3, 5}
- `gamma` ∈ {1.3, 1.5}
- `mask_type` ∈ {sparsemax, entmax}
- `lambda_sparse` ∈ {1e-4, 1e-3}

48 kombinasyon fazla olacağından iki aşamalı yapılır:
1. **Aşama 1 (12 koşum):** `mask_type="sparsemax"`, `lambda_sparse=1e-3` sabit, `n_d/n_a × n_steps × gamma` taranır.
2. **Aşama 2 (8 koşum):** Aşama 1'in en iyi 2 konfigürasyonunda `mask_type × lambda_sparse` (2×2) taranır.

Seçim kriteri: validasyon AUC'si (erken durdurma da AUC'yi izler). Ayarlar: `max_epochs=100`,
`patience=20`, `batch_size=1024`, `virtual_batch_size=128`.

In [6]:
train_df = load_split("train", VARIANT)
val_df = load_split("val", VARIANT)
X_train_raw, y_train = to_xy(train_df)
X_val_raw, y_val = to_xy(val_df)

preproc = make_preprocessor("tabnet")
X_train = encode_categoricals(preproc.fit_transform(X_train_raw))[FEATURES].to_numpy(dtype="float32")
X_val = encode_categoricals(preproc.transform(X_val_raw))[FEATURES].to_numpy(dtype="float32")
y_train = y_train.to_numpy()
y_val = y_val.to_numpy()

X_train.shape, X_val.shape

((20975, 23), (4495, 23))

In [7]:
import time

import numpy as np
import pandas as pd


def run_tabnet(n_d, n_steps, gamma, mask_type, lambda_sparse):
    """Train one TabNet config, return its best validation AUC/logloss and timing."""
    start = time.time()
    clf = TabNetClassifier(
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=2,
        n_d=n_d,
        n_a=n_d,
        n_steps=n_steps,
        gamma=gamma,
        mask_type=mask_type,
        lambda_sparse=lambda_sparse,
        seed=SEED,
        verbose=0,
    )
    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_name=["val"],
        eval_metric=["auc", "logloss"],
        max_epochs=100,
        patience=20,
        batch_size=1024,
        virtual_batch_size=128,
    )
    elapsed = time.time() - start
    best_epoch = int(np.argmax(clf.history["val_auc"]))
    return {
        "n_d": n_d,
        "n_a": n_d,
        "n_steps": n_steps,
        "gamma": gamma,
        "mask_type": mask_type,
        "lambda_sparse": lambda_sparse,
        "val_auc": clf.history["val_auc"][best_epoch],
        "val_logloss": clf.history["val_logloss"][best_epoch],
        "epochs_trained": len(clf.history["val_auc"]),
        "train_seconds": round(elapsed, 1),
    }

### Aşama 1: `n_d/n_a × n_steps × gamma` (12 koşum, mask/lambda sabit)

In [8]:
import itertools

FIXED_MASK_TYPE = "sparsemax"
FIXED_LAMBDA_SPARSE = 1e-3

stage1_grid = list(itertools.product([8, 16, 32], [3, 5], [1.3, 1.5]))
stage1_results = []
for i, (n_d, n_steps, gamma) in enumerate(stage1_grid, start=1):
    print(f"[Aşama 1] {i}/{len(stage1_grid)}  n_d=n_a={n_d} n_steps={n_steps} gamma={gamma}")
    row = run_tabnet(n_d, n_steps, gamma, FIXED_MASK_TYPE, FIXED_LAMBDA_SPARSE)
    row["stage"] = 1
    stage1_results.append(row)
    print("  val_auc =", round(row["val_auc"], 4), " epochs =", row["epochs_trained"])

stage1_df = pd.DataFrame(stage1_results).sort_values("val_auc", ascending=False)
stage1_df

[Aşama 1] 1/12  n_d=n_a=8 n_steps=3 gamma=1.3



Early stopping occurred at epoch 74 with best_epoch = 54 and best_val_logloss = 0.43212


  val_auc = 0.7801  epochs = 75
[Aşama 1] 2/12  n_d=n_a=8 n_steps=3 gamma=1.5



Early stopping occurred at epoch 70 with best_epoch = 50 and best_val_logloss = 0.43807


  val_auc = 0.771  epochs = 71
[Aşama 1] 3/12  n_d=n_a=8 n_steps=5 gamma=1.3


...


Early stopping occurred at epoch 54 with best_epoch = 34 and best_val_logloss = 0.43729


  val_auc = 0.7717  epochs = 55
[Aşama 1] 12/12  n_d=n_a=32 n_steps=5 gamma=1.5



Early stopping occurred at epoch 61 with best_epoch = 41 and best_val_logloss = 0.43984


  val_auc = 0.7681  epochs = 62


,n_d,n_a,n_steps,gamma,mask_type,lambda_sparse,val_auc,val_logloss,epochs_trained,train_seconds,stage
0,8,8,3,1.3,sparsemax,0.001,0.780148,0.432118,75,70.2,1
5,16,16,3,1.5,sparsemax,0.001,0.778712,0.434481,49,49.8,1
8,32,32,3,1.3,sparsemax,0.001,0.777008,0.435760,48,56.7,1
4,16,16,3,1.3,sparsemax,0.001,0.775547,0.438140,64,64.8,1
9,32,32,3,1.5,sparsemax,0.001,0.774172,0.435883,77,93.1,1
10,32,32,5,1.3,sparsemax,0.001,0.771658,0.437751,55,113.9,1
1,8,8,3,1.5,sparsemax,0.001,0.771010,0.438068,71,68.4,1
7,16,16,5,1.5,sparsemax,0.001,0.769731,0.437589,84,122.1,1
11,32,32,5,1.5,sparsemax,0.001,0.768081,0.440161,62,116.0,1
6,16,16,5,1.3,sparsemax,0.001,0.766425,0.439541,45,68.5,1


### Aşama 2: en iyi 2 konfigürasyonda `mask_type × lambda_sparse` (2 taban × 4 kombinasyon = 8; sparsemax/1e-3 olan ikisi Aşama 1'den tekrar kullanılır, 6 yeni koşum yapılır)

In [9]:
top2 = stage1_df.head(2)[["n_d", "n_steps", "gamma"]].to_dict("records")

stage2_results = []
mask_lambda_grid = list(itertools.product(["sparsemax", "entmax"], [1e-4, 1e-3]))
for base in top2:
    for mask_type, lambda_sparse in mask_lambda_grid:
        if mask_type == FIXED_MASK_TYPE and lambda_sparse == FIXED_LAMBDA_SPARSE:
            continue  # already have this one from stage 1
        print(f"[Aşama 2] base={base} mask_type={mask_type} lambda_sparse={lambda_sparse}")
        row = run_tabnet(base["n_d"], base["n_steps"], base["gamma"], mask_type, lambda_sparse)
        row["stage"] = 2
        stage2_results.append(row)
        print("  val_auc =", round(row["val_auc"], 4), " epochs =", row["epochs_trained"])

stage2_df = pd.DataFrame(stage2_results)
stage2_df

[Aşama 2] base={'n_d': 8, 'n_steps': 3, 'gamma': 1.3} mask_type=sparsemax lambda_sparse=0.0001



Early stopping occurred at epoch 40 with best_epoch = 20 and best_val_logloss = 0.43722


  val_auc = 0.7713  epochs = 41
[Aşama 2] base={'n_d': 8, 'n_steps': 3, 'gamma': 1.3} mask_type=entmax lambda_sparse=0.0001



Early stopping occurred at epoch 44 with best_epoch = 24 and best_val_logloss = 0.43308


  val_auc = 0.7762  epochs = 45
[Aşama 2] base={'n_d': 8, 'n_steps': 3, 'gamma': 1.3} mask_type=entmax lambda_sparse=0.001


...


Early stopping occurred at epoch 38 with best_epoch = 18 and best_val_logloss = 0.43734


  val_auc = 0.7703  epochs = 39
[Aşama 2] base={'n_d': 16, 'n_steps': 3, 'gamma': 1.5} mask_type=entmax lambda_sparse=0.001



Early stopping occurred at epoch 40 with best_epoch = 20 and best_val_logloss = 0.43637


  val_auc = 0.7747  epochs = 41


,n_d,n_a,n_steps,gamma,mask_type,lambda_sparse,val_auc,val_logloss,epochs_trained,train_seconds,stage
0,8,8,3,1.3,sparsemax,0.0001,0.771336,0.437222,41,40.3,2
1,8,8,3,1.3,entmax,0.0001,0.776157,0.434184,45,44.0,2
2,8,8,3,1.3,entmax,0.0010,0.773971,0.436755,58,56.9,2
3,16,16,3,1.5,sparsemax,0.0001,0.777426,0.435797,65,67.4,2
4,16,16,3,1.5,entmax,0.0001,0.770254,0.443087,39,40.7,2
5,16,16,3,1.5,entmax,0.0010,0.774721,0.436408,41,46.0,2


In [10]:
from src.config import REPORTS_DIR

grid_df = pd.concat([stage1_df, stage2_df], ignore_index=True).sort_values(
    "val_auc", ascending=False
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
grid_path = REPORTS_DIR / "tabnet_grid.csv"
grid_df.to_csv(grid_path, index=False)

best = grid_df.iloc[0]
print(f"En iyi config -> n_d=n_a={best.n_d} n_steps={best.n_steps} gamma={best.gamma} "
      f"mask_type={best.mask_type} lambda_sparse={best.lambda_sparse} "
      f"(val_auc={best.val_auc:.4f})")
grid_df

En iyi config -> n_d=n_a=8 n_steps=3 gamma=1.3 mask_type=sparsemax lambda_sparse=0.001 (val_auc=0.7801)


,n_d,n_a,n_steps,gamma,mask_type,lambda_sparse,val_auc,val_logloss,epochs_trained,train_seconds,stage
0,8,8,3,1.3,sparsemax,0.0010,0.780148,0.432118,75,70.2,1
1,16,16,3,1.5,sparsemax,0.0010,0.778712,0.434481,49,49.8,1
15,16,16,3,1.5,sparsemax,0.0001,0.777426,0.435797,65,67.4,2
2,32,32,3,1.3,sparsemax,0.0010,0.777008,0.435760,48,56.7,1
13,8,8,3,1.3,entmax,0.0001,0.776157,0.434184,45,44.0,2
3,16,16,3,1.3,sparsemax,0.0010,0.775547,0.438140,64,64.8,1
17,16,16,3,1.5,entmax,0.0010,0.774721,0.436408,41,46.0,2
4,32,32,3,1.5,sparsemax,0.0010,0.774172,0.435883,77,93.1,1
14,8,8,3,1.3,entmax,0.0010,0.773971,0.436755,58,56.9,2
5,32,32,5,1.3,sparsemax,0.0010,0.771658,0.437751,55,113.9,1
